# Model Comparison

**Task IDs:** T-38  
**Purpose:** Compare all trained models across 5-fold stratified CV.
Table of PR-AUC, ROC-AUC, recall, precision, F1 (mean +/- std).
Bar chart + PR curves overlaid.

Findings recorded in `02-models/Model Comparison.md` (Obsidian vault).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

from src.config import get_path, load_config

warnings.filterwarnings("ignore", category=FutureWarning)
cfg = load_config()
mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
exp = mlflow.get_experiment_by_name(cfg["mlflow"]["experiment_name"])

runs = mlflow.search_runs([exp.experiment_id]) if exp else pd.DataFrame()
runs

In [ ]:
metric_cols = [c for c in runs.columns if c.startswith("metrics.") and not c.endswith("_std")]
display_cols = ["params.model"] + metric_cols
comparison = runs[display_cols].dropna(subset=["metrics.pr_auc"]).sort_values("metrics.pr_auc", ascending=False)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
models = comparison["params.model"].values
pr_auc = comparison["metrics.pr_auc"].values
roc_auc = comparison["metrics.roc_auc"].values

x = np.arange(len(models))
width = 0.35
ax.bar(x - width/2, pr_auc, width, label="PR-AUC", color="#3b82f6")
ax.bar(x + width/2, roc_auc, width, label="ROC-AUC", color="#f59e0b")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha="right")
ax.set_ylabel("Score")
ax.set_title("Model Comparison: PR-AUC vs ROC-AUC")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("../reports/figures/models/model_comparison_bar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
processed_dir = get_path(cfg, "processed_dir")
val = pd.read_parquet(processed_dir / "val.parquet")
X_val = val.drop(columns=["Class"])
y_val = val["Class"]

fig, ax = plt.subplots(figsize=(10, 8))
for model_name in comparison["params.model"].values:
    import joblib

    path = get_path(cfg, "models_dir") / f"{model_name}.joblib"
    if not path.exists():
        continue
    model = joblib.load(path)
    y_proba = model.predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, y_proba)
    ax.plot(recall, precision, label=model_name, alpha=0.8)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves (Validation Set)")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("../reports/figures/models/pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Conclusion

The best model by PR-AUC is selected per ADR-003. Record the comparison
table and PR curves in `02-models/Model Comparison.md`.